In [ ]:
%pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117

In [1]:
import torch
print(torch.__version__)
print(torch.cuda.get_device_name(0))

2.10.0+cu128
Tesla T4


In [2]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler

import torchvision.transforms as transforms
import torchvision.models as models
from tqdm import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
# Should print: Device: cuda

Device: cuda


In [3]:
set_seed(456) 

In [4]:
CELEBA_ROOT = "/kaggle/input/datasets/jessicali9530/celeba-dataset"
IMG_DIR     = os.path.join(CELEBA_ROOT, "img_align_celeba", "img_align_celeba")
ATTR_PATH   = os.path.join(CELEBA_ROOT, "list_attr_celeba.csv")
SPLIT_PATH  = os.path.join(CELEBA_ROOT, "list_eval_partition.csv")

attr_df  = pd.read_csv(ATTR_PATH)
split_df = pd.read_csv(SPLIT_PATH)

attr_df.columns  = attr_df.columns.str.strip()
split_df.columns = split_df.columns.str.strip()

# Merge on image_id to guarantee alignment
meta = attr_df[["image_id", "Blond_Hair", "Male"]].merge(
    split_df[["image_id", "partition"]], on="image_id"
)

meta["y"]     = (meta["Blond_Hair"] == 1).astype(int)  # 1 = blond
meta["place"] = (meta["Male"] == 1).astype(int)        # 1 = male
meta["split"] = meta["partition"]                       # 0=train 1=val 2=test

# group = 2*y + place
# 0: non-blond female (majority)
# 1: non-blond male   (majority)
# 2: blond female     (majority)
# 3: blond male       (CRITICAL MINORITY — only ~1,400 train samples)
meta["group"] = 2 * meta["y"] + meta["place"]

META_SAVE = "/kaggle/working/celeba_metadata.csv"
meta.to_csv(META_SAVE, index=False)

# Sanity check — print group distribution on train split
train_meta = meta[meta["split"] == 0]
print("Train group distribution:")
print(train_meta["group"].value_counts().sort_index())
print(f"\nTotal train: {len(train_meta)}")
print(f"Group 3 (blond+male, minority): {(train_meta['group']==3).sum()} samples")
# Expected: Group 0 ~71k, Group 1 ~66k, Group 2 ~22k, Group 3 ~1.4k
# If Group 3 < 500, something is wrong with the group definition — stop here

Train group distribution:
group
0    71629
1    66874
2    22880
3     1387
Name: count, dtype: int64

Total train: 162770
Group 3 (blond+male, minority): 1387 samples


In [5]:
class CelebADataset(Dataset):
    def __init__(self, meta_path, img_dir, split_name, transform=None):
        self.img_dir   = img_dir
        self.transform = transform

        meta = pd.read_csv(meta_path)
        split_map = {"train": 0, "val": 1, "test": 2}
        self.data = meta[meta["split"] == split_map[split_name]].reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row   = self.data.iloc[idx]
        img   = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(row["y"]), int(row["group"]), row["image_id"]

In [6]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop((224, 224), scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_data = CelebADataset(META_SAVE, IMG_DIR, "train", transform=train_transform)
val_data   = CelebADataset(META_SAVE, IMG_DIR, "val",   transform=eval_transform)
test_data  = CelebADataset(META_SAVE, IMG_DIR, "test",  transform=eval_transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_data,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
# Expected: Train: 162770 | Val: 19867 | Test: 19962

Train: 162770 | Val: 19867 | Test: 19962


In [7]:
def make_resnet50():
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 2)  # no dropout, same as your WB setup
    return m.to(device)

erm_model = make_resnet50()

EPOCHS    = 25
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(erm_model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

print(f"Training ERM on full CelebA train set ({len(train_data)} samples) for {EPOCHS} epochs")
print(f"Estimated time on P100: ~6–8 hours")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 182MB/s] 


Training ERM on full CelebA train set (162770 samples) for 25 epochs
Estimated time on P100: ~6–8 hours


In [7]:
print("Starting CelebA ERM Training...")

best_val_wga = 0.0

for epoch in range(EPOCHS):
    erm_model.train()
    train_loss, correct, total = 0.0, 0, 0
    loop = tqdm(train_loader, desc=f"ERM Epoch [{epoch+1}/{EPOCHS}]")

    for images, targets, groups, _ in loop:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = erm_model(images)
        loss   = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == targets).sum().item()
        total      += targets.size(0)
        loop.set_postfix({"loss": train_loss / (loop.n + 1), "acc": correct / total})

    # Validation
    erm_model.eval()
    group_correct = {g: 0 for g in range(4)}
    group_total   = {g: 0 for g in range(4)}
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, targets, groups, _ in val_loader:
            images, targets = images.to(device), targets.to(device)
            logits = erm_model(images)
            preds  = torch.argmax(logits, dim=1)
            val_correct += (preds == targets).sum().item()
            val_total   += targets.size(0)
            for i in range(len(targets)):
                g = groups[i].item()
                group_total[g]   += 1
                group_correct[g] += (preds[i] == targets[i]).item()

    group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in range(4)}
    wga     = min(group_accs.values())
    avg_acc = val_correct / val_total

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Acc: {correct/total:.4f} "
          f"| Val Acc: {avg_acc:.4f} | Val WGA: {wga:.4f} "
          f"| G0:{group_accs[0]:.3f} G1:{group_accs[1]:.3f} "
          f"G2:{group_accs[2]:.3f} G3:{group_accs[3]:.3f}")

    # Save best checkpoint by WGA (not avg acc — this is intentional)
    if wga > best_val_wga:
        best_val_wga = wga
        torch.save(erm_model.state_dict(), "/kaggle/working/celeba_erm_best.pth")
        print(f"  → New best WGA: {wga:.4f} — checkpoint saved")

# Also save final epoch regardless
torch.save(erm_model.state_dict(), "/kaggle/working/celeba_erm_final.pth")
print(f"\nERM training complete. Best val WGA: {best_val_wga:.4f}")

Starting CelebA ERM Training...


ERM Epoch [1/25]: 100%|██████████| 2544/2544 [14:03<00:00,  3.01it/s, loss=0.126, acc=0.948]


Epoch 01/25 | Train Acc: 0.9485 | Val Acc: 0.9548 | Val WGA: 0.4176 | G0:0.944 G1:0.995 G2:0.905 G3:0.418
  → New best WGA: 0.4176 — checkpoint saved


ERM Epoch [2/25]: 100%|██████████| 2544/2544 [12:52<00:00,  3.29it/s, loss=0.106, acc=0.957]


Epoch 02/25 | Train Acc: 0.9570 | Val Acc: 0.9564 | Val WGA: 0.2912 | G0:0.956 G1:0.997 G2:0.882 G3:0.291


...

Epoch 23/25 | Train Acc: 0.9927 | Val Acc: 0.9514 | Val WGA: 0.4615 | G0:0.951 G1:0.992 G2:0.868 G3:0.462


ERM Epoch [24/25]: 100%|██████████| 2544/2544 [12:51<00:00,  3.30it/s, loss=0.019, acc=0.993] 


Epoch 24/25 | Train Acc: 0.9927 | Val Acc: 0.9517 | Val WGA: 0.4121 | G0:0.947 G1:0.993 G2:0.881 G3:0.412


ERM Epoch [25/25]: 100%|██████████| 2544/2544 [12:50<00:00,  3.30it/s, loss=0.0181, acc=0.993]


Epoch 25/25 | Train Acc: 0.9935 | Val Acc: 0.9511 | Val WGA: 0.3901 | G0:0.952 G1:0.993 G2:0.862 G3:0.390

ERM training complete. Best val WGA: 0.5385


In [8]:
# Load best checkpoint for test eval
erm_model.load_state_dict(torch.load("/kaggle/working/celeba_erm_best.pth"))
erm_model.eval()

group_correct = {g: 0 for g in range(4)}
group_total   = {g: 0 for g in range(4)}
test_correct, test_total = 0, 0
logs = []

with torch.no_grad():
    for images, targets, groups, img_ids in tqdm(test_loader, desc="ERM Test Eval"):
        images, targets = images.to(device), targets.to(device)
        logits = erm_model(images)
        probs  = torch.softmax(logits, dim=1)
        preds  = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total   += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g]   += 1
            group_correct[g] += (preds[i] == targets[i]).item()
            logs.append({
                "image_id":     img_ids[i],
                "label":        targets[i].item(),
                "group":        g,
                "prediction":   preds[i].item(),
                "confidence":   probs[i][preds[i]].item(),
                "logit_class_0": logits[i][0].item(),
                "logit_class_1": logits[i][1].item(),
            })

group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in range(4)}
test_wga   = min(group_accs.values())
test_avg   = test_correct / test_total

print("\n=== CelebA ERM TEST RESULTS ===")
print(f"Avg Accuracy : {test_avg:.4f}")
print(f"WGA          : {test_wga:.4f}  ← should be 45–55% if training worked correctly")
print(f"Group 0 (non-blond female) : {group_accs[0]:.4f}")
print(f"Group 1 (non-blond male)   : {group_accs[1]:.4f}")
print(f"Group 2 (blond female)     : {group_accs[2]:.4f}")
print(f"Group 3 (blond male)       : {group_accs[3]:.4f}  ← minority, will be lowest")

pd.DataFrame(logs).to_csv("/kaggle/working/celeba_erm_test_predictions.csv", index=False)
print("\nSaved: celeba_erm_test_predictions.csv")

ERM Test Eval: 100%|██████████| 312/312 [00:57<00:00,  5.45it/s]



=== CelebA ERM TEST RESULTS ===
Avg Accuracy : 0.9487
WGA          : 0.5833  ← should be 45–55% if training worked correctly
Group 0 (non-blond female) : 0.9519
Group 1 (non-blond male)   : 0.9841
Group 2 (blond female)     : 0.8552
Group 3 (blond male)       : 0.5833  ← minority, will be lowest

Saved: celeba_erm_test_predictions.csv


In [20]:
# 95/5 split — fixed seed for reproducibility
total_train   = len(train_data)
split_95_len  = int(0.95 * total_train)
split_5_len   = total_train - split_95_len

train_95_ds, train_5_ds = random_split(
    train_data,
    [split_95_len, split_5_len],
    generator=torch.Generator().manual_seed(42)
)

print(f"Phase 1 backbone set : {split_95_len} samples")
print(f"Phase 2 head set     : {split_5_len} samples")

# Verify group 3 representation in the 5% set before proceeding
targets_5 = [train_data[i][1] for i in train_5_ds.indices]
groups_5  = [train_data[i][2] for i in train_5_ds.indices]
from collections import Counter
print("\n5% split group distribution:", Counter(groups_5))
print("5% split class distribution:", Counter(targets_5))
# Group 3 should have at least ~50 samples — if fewer, the head retraining will be noisy

train_95_loader = DataLoader(train_95_ds, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)

fl_model  = make_resnet50()
optimizer_95 = optim.SGD(fl_model.parameters(), lr=1e-3, momentum=0.9, weight_decay=1e-4)

print(f"\n[Phase 1] Training backbone on {split_95_len} samples for {EPOCHS} epochs")
print("Estimated time on P100: ~5–7 hours")

Phase 1 backbone set : 154631 samples
Phase 2 head set     : 8139 samples

5% split group distribution: Counter({0: 3581, 1: 3418, 2: 1067, 3: 73})
5% split class distribution: Counter({0: 6999, 1: 1140})

[Phase 1] Training backbone on 154631 samples for 25 epochs
Estimated time on P100: ~5–7 hours


In [14]:
for epoch in range(EPOCHS):
    fl_model.train()
    total_loss, correct, total = 0.0, 0, 0
    loop = tqdm(train_95_loader, desc=f"FL Backbone Epoch [{epoch+1}/{EPOCHS}]")

    for images, targets, _, _ in loop:
        images, targets = images.to(device), targets.to(device)
        optimizer_95.zero_grad()
        logits = fl_model(images)
        loss   = criterion(logits, targets)
        loss.backward()
        optimizer_95.step()

        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == targets).sum().item()
        total      += targets.size(0)
        loop.set_postfix({"loss": total_loss / (loop.n + 1), "acc": correct / total})

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_95_loader):.4f} | Acc: {correct/total:.4f}")

torch.save(fl_model.state_dict(), "/kaggle/working/celeba_fl_backbone_123.pth")
print("Phase 1 complete. Backbone saved.")

FL Backbone Epoch [1/25]:  32%|███▏      | 783/2417 [08:15<17:14,  1.58it/s, loss=0.146, acc=0.94] 


KeyboardInterrupt: 

In [21]:
fl_model.load_state_dict(torch.load("/kaggle/input/models/ankitaanand26/celeba-fl-backbone/pytorch/default/1/celeba_fl_backbone (1).pth"))


<All keys matched successfully>

In [22]:
print("\n[Phase 2] Freezing backbone, retraining classifier head on class-balanced 5% split")

# Freeze all layers except fc
for param in fl_model.parameters():
    param.requires_grad = False

in_features = fl_model.fc.in_features
fl_model.fc = nn.Linear(in_features, 2).to(device)  # fresh head
optimizer_fc = optim.SGD(fl_model.fc.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-4)
# Note: lr=1e-2 for head-only retraining (10x higher than backbone) — standard for DFR-style methods

# Class-balanced sampler on the 5% set
targets_5_arr   = np.array(targets_5)
class_counts    = np.bincount(targets_5_arr)
class_weights   = 1.0 / class_counts
sample_weights  = np.array([class_weights[y] for y in targets_5_arr])

sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(sample_weights),
    replacement=True
)

train_5_loader = DataLoader(
    train_5_ds, batch_size=32, sampler=sampler, num_workers=4, pin_memory=True
)

print(f"Class counts in 5% set — Class 0: {class_counts[0]}, Class 1: {class_counts[1]}")
print(f"Sampler will balance these to 50/50 within each batch")

# Head retraining: 50 epochs (not steps — cleaner and more reproducible than step counting)
HEAD_EPOCHS = 50
fl_model.train()

for epoch in range(HEAD_EPOCHS):
    correct, total, total_loss = 0, 0, 0.0
    for images, targets, _, _ in train_5_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer_fc.zero_grad()
        logits = fl_model(images)
        loss   = criterion(logits, targets)
        loss.backward()
        optimizer_fc.step()
        total_loss += loss.item()
        preds       = torch.argmax(logits, dim=1)
        correct    += (preds == targets).sum().item()
        total      += targets.size(0)

    if (epoch + 1) % 1 == 0:
        print(f"Head epoch {epoch+1}/{HEAD_EPOCHS} | Loss: {total_loss/len(train_5_loader):.4f} | Acc: {correct/total:.4f}")

torch.save(fl_model.state_dict(), "/kaggle/working/celeba_fl_final_456.pth")
print("Phase 2 complete. FL model saved.")


[Phase 2] Freezing backbone, retraining classifier head on class-balanced 5% split
Class counts in 5% set — Class 0: 6999, Class 1: 1140
Sampler will balance these to 50/50 within each batch
Head epoch 1/50 | Loss: 0.2347 | Acc: 0.9147
Head epoch 2/50 | Loss: 0.2191 | Acc: 0.9214
Head epoch 3/50 | Loss: 0.2044 | Acc: 0.9321
Head epoch 4/50 | Loss: 0.2043 | Acc: 0.9233
Head epoch 5/50 | Loss: 0.1885 | Acc: 0.9329
Head epoch 6/50 | Loss: 0.2042 | Acc: 0.9294
Head epoch 7/50 | Loss: 0.1800 | Acc: 0.9327
Head epoch 8/50 | Loss: 0.2142 | Acc: 0.9318
Head epoch 9/50 | Loss: 0.2081 | Acc: 0.9308
Head epoch 10/50 | Loss: 0.2353 | Acc: 0.9233
Head epoch 11/50 | Loss: 0.1634 | Acc: 0.9394
Head epoch 12/50 | Loss: 0.1940 | Acc: 0.9303
Head epoch 13/50 | Loss: 0.1898 | Acc: 0.9306
Head epoch 14/50 | Loss: 0.1833 | Acc: 0.9330
Head epoch 15/50 | Loss: 0.1801 | Acc: 0.9361
Head epoch 16/50 | Loss: 0.1532 | Acc: 0.9431
Head epoch 17/50 | Loss: 0.1713 | Acc: 0.9411
Head epoch 18/50 | Loss: 0.2020 | A

In [23]:
fl_model.eval()

group_correct = {g: 0 for g in range(4)}
group_total   = {g: 0 for g in range(4)}
test_correct, test_total = 0, 0
fl_logs = []

with torch.no_grad():
    for images, targets, groups, img_ids in tqdm(test_loader, desc="FL Test Eval"):
        images, targets = images.to(device), targets.to(device)
        logits = fl_model(images)
        probs  = torch.softmax(logits, dim=1)
        preds  = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total   += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g]   += 1
            group_correct[g] += (preds[i] == targets[i]).item()
            fl_logs.append({
                "image_id":      img_ids[i],
                "label":         targets[i].item(),
                "group":         g,
                "fl_prediction": preds[i].item(),
                "fl_confidence": probs[i][preds[i]].item(),
                "fl_logit_0":    logits[i][0].item(),
                "fl_logit_1":    logits[i][1].item(),
            })

group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in range(4)}
fl_wga     = min(group_accs.values())
fl_avg     = test_correct / test_total

print("\n=== CelebA FL TEST RESULTS ===")
print(f"Avg Accuracy : {fl_avg:.4f}")
print(f"WGA          : {fl_wga:.4f}  ← should be noticeably higher than ERM WGA")
print(f"Group 0: {group_accs[0]:.4f} | Group 1: {group_accs[1]:.4f} "
      f"| Group 2: {group_accs[2]:.4f} | Group 3: {group_accs[3]:.4f}")

pd.DataFrame(fl_logs).to_csv("/kaggle/working/celeba_fl_test_predictions.csv", index=False)
print("\nSaved: celeba_fl_test_predictions.csv")

FL Test Eval: 100%|██████████| 312/312 [01:03<00:00,  4.93it/s]


=== CelebA FL TEST RESULTS ===
Avg Accuracy : 0.9145
WGA          : 0.6944  ← should be noticeably higher than ERM WGA
Group 0: 0.8700 | Group 1: 0.9639 | Group 2: 0.9560 | Group 3: 0.6944

Saved: celeba_fl_test_predictions.csv


Seed 123 ERM

In [8]:
from torch.amp import autocast, GradScaler

scaler = GradScaler(device="cuda")

best_val_wga = 0.0

for epoch in range(EPOCHS):
    erm_model.train()
    train_loss, correct, total = 0.0, 0, 0
    loop = tqdm(train_loader, desc=f"ERM Epoch [{epoch+1}/{EPOCHS}]")

    for images, targets, groups, _ in loop:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # MIXED PRECISION
        
        with autocast(device_type="cuda"):
            logits = erm_model(images)
            loss   = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

        loop.set_postfix({
            "loss": train_loss / (loop.n + 1),
            "acc": correct / total
        })

    # ---------------- VALIDATION ----------------
    erm_model.eval()
    group_correct = {g: 0 for g in range(4)}
    group_total   = {g: 0 for g in range(4)}
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, targets, groups, _ in val_loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            # AMP even in eval (faster)
            with autocast(device_type="cuda"):
                logits = erm_model(images)

            preds = torch.argmax(logits, dim=1)

            val_correct += (preds == targets).sum().item()
            val_total += targets.size(0)

            for i in range(len(targets)):
                g = groups[i].item()
                group_total[g] += 1
                group_correct[g] += (preds[i] == targets[i]).item()

    group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in range(4)}
    wga = min(group_accs.values())
    avg_acc = val_correct / val_total

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Acc: {correct/total:.4f} "
          f"| Val Acc: {avg_acc:.4f} | Val WGA: {wga:.4f} "
          f"| G0:{group_accs[0]:.3f} G1:{group_accs[1]:.3f} "
          f"G2:{group_accs[2]:.3f} G3:{group_accs[3]:.3f}")

    if wga > best_val_wga:
        best_val_wga = wga
        torch.save(erm_model.state_dict(), "/kaggle/working/celeba_erm_best.pth")
        print(f"  → New best WGA: {wga:.4f} — checkpoint saved")

torch.save(erm_model.state_dict(), "/kaggle/working/celeba_erm_final.pth")
print(f"\nERM training complete. Best val WGA: {best_val_wga:.4f}")

ERM Epoch [1/25]: 100%|██████████| 2544/2544 [12:03<00:00,  3.52it/s, loss=0.126, acc=0.949]


Epoch 01/25 | Train Acc: 0.9485 | Val Acc: 0.9531 | Val WGA: 0.4176 | G0:0.940 G1:0.995 G2:0.906 G3:0.418
  → New best WGA: 0.4176 — checkpoint saved


ERM Epoch [2/25]: 100%|██████████| 2544/2544 [11:46<00:00,  3.60it/s, loss=0.106, acc=0.956]


Epoch 02/25 | Train Acc: 0.9563 | Val Acc: 0.9571 | Val WGA: 0.4011 | G0:0.956 G1:0.996 G2:0.882 G3:0.401


ERM Epoch [3/25]: 100%|██████████| 2544/2544 [11:45<00:00,  3.61it/s, loss=0.1, acc=0.959]   


...

Epoch 23/25 | Train Acc: 0.9922 | Val Acc: 0.9501 | Val WGA: 0.2857 | G0:0.957 G1:0.997 G2:0.838 G3:0.286


ERM Epoch [24/25]: 100%|██████████| 2544/2544 [11:41<00:00,  3.63it/s, loss=0.0192, acc=0.993]


Epoch 24/25 | Train Acc: 0.9930 | Val Acc: 0.9510 | Val WGA: 0.4231 | G0:0.959 G1:0.993 G2:0.839 G3:0.423


ERM Epoch [25/25]: 100%|██████████| 2544/2544 [11:44<00:00,  3.61it/s, loss=0.0186, acc=0.993]


Epoch 25/25 | Train Acc: 0.9931 | Val Acc: 0.9497 | Val WGA: 0.4341 | G0:0.946 G1:0.992 G2:0.872 G3:0.434

ERM training complete. Best val WGA: 0.5769


In [9]:
# Load best checkpoint for test eval
erm_model.load_state_dict(torch.load("/kaggle/working/celeba_erm_best.pth"))
erm_model.eval()

group_correct = {g: 0 for g in range(4)}
group_total   = {g: 0 for g in range(4)}
test_correct, test_total = 0, 0
logs = []

with torch.no_grad():
    for images, targets, groups, img_ids in tqdm(test_loader, desc="ERM Test Eval"):
        images, targets = images.to(device), targets.to(device)
        logits = erm_model(images)
        probs  = torch.softmax(logits, dim=1)
        preds  = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total   += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g]   += 1
            group_correct[g] += (preds[i] == targets[i]).item()
            logs.append({
                "image_id":     img_ids[i],
                "label":        targets[i].item(),
                "group":        g,
                "prediction":   preds[i].item(),
                "confidence":   probs[i][preds[i]].item(),
                "logit_class_0": logits[i][0].item(),
                "logit_class_1": logits[i][1].item(),
            })

group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in range(4)}
test_wga   = min(group_accs.values())
test_avg   = test_correct / test_total

print("\n=== CelebA ERM TEST RESULTS ===")
print(f"Avg Accuracy : {test_avg:.4f}")
print(f"WGA          : {test_wga:.4f}  ← should be 45–55% if training worked correctly")
print(f"Group 0 (non-blond female) : {group_accs[0]:.4f}")
print(f"Group 1 (non-blond male)   : {group_accs[1]:.4f}")
print(f"Group 2 (blond female)     : {group_accs[2]:.4f}")
print(f"Group 3 (blond male)       : {group_accs[3]:.4f}  ← minority, will be lowest")

pd.DataFrame(logs).to_csv("/kaggle/working/celeba_erm_test_predictions.csv", index=False)
print("\nSaved: celeba_erm_test_predictions.csv")

ERM Test Eval: 100%|██████████| 312/312 [01:11<00:00,  4.38it/s]


=== CelebA ERM TEST RESULTS ===
Avg Accuracy : 0.9524
WGA          : 0.5500  ← should be 45–55% if training worked correctly
Group 0 (non-blond female) : 0.9488
Group 1 (non-blond male)   : 0.9863
Group 2 (blond female)     : 0.8923
Group 3 (blond male)       : 0.5500  ← minority, will be lowest

Saved: celeba_erm_test_predictions.csv


In [8]:
from torch.amp import autocast, GradScaler

scaler = GradScaler(device="cuda")

best_val_wga = 0.0

for epoch in range(EPOCHS):
    erm_model.train()
    train_loss, correct, total = 0.0, 0, 0
    loop = tqdm(train_loader, desc=f"ERM Epoch [{epoch+1}/{EPOCHS}]")

    for images, targets, groups, _ in loop:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # MIXED PRECISION
        
        with autocast(device_type="cuda"):
            logits = erm_model(images)
            loss   = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

        loop.set_postfix({
            "loss": train_loss / (loop.n + 1),
            "acc": correct / total
        })

    # ---------------- VALIDATION ----------------
    erm_model.eval()
    group_correct = {g: 0 for g in range(4)}
    group_total   = {g: 0 for g in range(4)}
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, targets, groups, _ in val_loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            # AMP even in eval (faster)
            with autocast(device_type="cuda"):
                logits = erm_model(images)

            preds = torch.argmax(logits, dim=1)

            val_correct += (preds == targets).sum().item()
            val_total += targets.size(0)

            for i in range(len(targets)):
                g = groups[i].item()
                group_total[g] += 1
                group_correct[g] += (preds[i] == targets[i]).item()

    group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in range(4)}
    wga = min(group_accs.values())
    avg_acc = val_correct / val_total

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Acc: {correct/total:.4f} "
          f"| Val Acc: {avg_acc:.4f} | Val WGA: {wga:.4f} "
          f"| G0:{group_accs[0]:.3f} G1:{group_accs[1]:.3f} "
          f"G2:{group_accs[2]:.3f} G3:{group_accs[3]:.3f}")

    if wga > best_val_wga:
        best_val_wga = wga
        torch.save(erm_model.state_dict(), "/kaggle/working/celeba_erm_best_seed456.pth")
        print(f"  → New best WGA: {wga:.4f} — checkpoint saved")

torch.save(erm_model.state_dict(), "/kaggle/working/celeba_erm_final_seed456.pth")
print(f"\nERM training complete. Best val WGA: {best_val_wga:.4f}")

ERM Epoch [1/25]: 100%|██████████| 2544/2544 [10:27<00:00,  4.05it/s, loss=0.126, acc=0.948]


Epoch 01/25 | Train Acc: 0.9483 | Val Acc: 0.9548 | Val WGA: 0.2692 | G0:0.960 G1:0.997 G2:0.862 G3:0.269
  → New best WGA: 0.2692 — checkpoint saved


ERM Epoch [2/25]: 100%|██████████| 2544/2544 [10:30<00:00,  4.03it/s, loss=0.107, acc=0.956]


Epoch 02/25 | Train Acc: 0.9561 | Val Acc: 0.9557 | Val WGA: 0.5000 | G0:0.944 G1:0.994 G2:0.908 G3:0.500
  → New best WGA: 0.5000 — checkpoint saved


ERM Epoch [3/25]: 100%|██████████| 2544/2544 [10:30<00:00,  4.03it/s, loss=0.1, acc=0.959]   


...

Epoch 23/25 | Train Acc: 0.9925 | Val Acc: 0.9488 | Val WGA: 0.3791 | G0:0.953 G1:0.991 G2:0.848 G3:0.379


ERM Epoch [24/25]: 100%|██████████| 2544/2544 [10:30<00:00,  4.04it/s, loss=0.0191, acc=0.993]


Epoch 24/25 | Train Acc: 0.9929 | Val Acc: 0.9502 | Val WGA: 0.4945 | G0:0.944 G1:0.992 G2:0.877 G3:0.495


ERM Epoch [25/25]: 100%|██████████| 2544/2544 [10:29<00:00,  4.04it/s, loss=0.0181, acc=0.993]


Epoch 25/25 | Train Acc: 0.9932 | Val Acc: 0.9485 | Val WGA: 0.3791 | G0:0.952 G1:0.993 G2:0.846 G3:0.379

ERM training complete. Best val WGA: 0.5604


In [9]:
# Load best checkpoint for test eval
erm_model.load_state_dict(torch.load("/kaggle/working/celeba_erm_best_seed456.pth"))
erm_model.eval()

group_correct = {g: 0 for g in range(4)}
group_total   = {g: 0 for g in range(4)}
test_correct, test_total = 0, 0
logs = []

with torch.no_grad():
    for images, targets, groups, img_ids in tqdm(test_loader, desc="ERM Test Eval"):
        images, targets = images.to(device), targets.to(device)
        logits = erm_model(images)
        probs  = torch.softmax(logits, dim=1)
        preds  = torch.argmax(logits, dim=1)

        test_correct += (preds == targets).sum().item()
        test_total   += targets.size(0)

        for i in range(len(targets)):
            g = groups[i].item()
            group_total[g]   += 1
            group_correct[g] += (preds[i] == targets[i]).item()
            logs.append({
                "image_id":     img_ids[i],
                "label":        targets[i].item(),
                "group":        g,
                "prediction":   preds[i].item(),
                "confidence":   probs[i][preds[i]].item(),
                "logit_class_0": logits[i][0].item(),
                "logit_class_1": logits[i][1].item(),
            })

group_accs = {g: group_correct[g] / (group_total[g] + 1e-8) for g in range(4)}
test_wga   = min(group_accs.values())
test_avg   = test_correct / test_total

print("\n=== CelebA ERM TEST RESULTS ===")
print(f"Avg Accuracy : {test_avg:.4f}")
print(f"WGA          : {test_wga:.4f}  ← should be 45–55% if training worked correctly")
print(f"Group 0 (non-blond female) : {group_accs[0]:.4f}")
print(f"Group 1 (non-blond male)   : {group_accs[1]:.4f}")
print(f"Group 2 (blond female)     : {group_accs[2]:.4f}")
print(f"Group 3 (blond male)       : {group_accs[3]:.4f}  ← minority, will be lowest")

pd.DataFrame(logs).to_csv("/kaggle/working/celeba_erm_test_predictions_seed456.csv", index=False)
print("\nSaved: celeba_erm_test_predictions_seed456.csv")

ERM Test Eval: 100%|██████████| 312/312 [01:10<00:00,  4.45it/s]


=== CelebA ERM TEST RESULTS ===
Avg Accuracy : 0.9507
WGA          : 0.5333  ← should be 45–55% if training worked correctly
Group 0 (non-blond female) : 0.9509
Group 1 (non-blond male)   : 0.9881
Group 2 (blond female)     : 0.8665
Group 3 (blond male)       : 0.5333  ← minority, will be lowest

Saved: celeba_erm_test_predictions_seed456.csv
